In [ ]:
from scipy.stats import chi2_contingency
# ==========================================
# 1. IMPUTACIÓN TEMPORAL (Solo en X_train)
# ==========================================
# Hacemos una copia para no contaminar los datos originales
df_eda = X_train.copy()
df_eda['deposit'] = y_train # Añadimos el target para ver su correlación también

# Supongamos que tu target es numérico o categórico. 
# Añádelo a la lista correspondiente para que el bucle lo evalúe.
# Ejemplo: num_cols = ['edad', 'salario', 'target'] o cat_cols = ['ciudad', 'color']
cat_atrib.append('deposit')
all_atrib = num_atrib + cat_atrib 

# Imputamos nulos para que las matemáticas no fallen
for col in num_atrib:
    if col in df_eda.columns:
        df_eda[col] = df_eda[col].fillna(df_eda[col].median())
        
for col in cat_atrib:
    if col in df_eda.columns:
        df_eda[col] = df_eda[col].fillna(df_eda[col].mode()[0])

# ==========================================
# 2. FUNCIONES ESTADÍSTICAS
# ==========================================
def cramers_v(x, y):
    """Cramér's V para dos variables categóricas."""
    confusion = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion)[0]
    n = confusion.sum().sum()
    r, k = confusion.shape
    # Evitar división por cero si alguna variable tiene una sola categoría
    if min(r, k) <= 1: 
        return 0.0
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

def correlation_ratio(categories, values):
    """Eta (η): correlación entre variable categórica y numérica."""
    categories = np.array(categories)
    values = np.array(values)
    grand_mean = values.mean()
    groups = [values[categories == c] for c in np.unique(categories)]
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total = sum((values - grand_mean)**2)
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else 0.0

# ==========================================
# 3. CREACIÓN DE LA MATRIZ DE CORRELACIÓN
# ==========================================
n_cols = len(all_atrib)
corr_matrix = pd.DataFrame(np.zeros((n_cols, n_cols)), index=all_atrib, columns=all_atrib)

for i, col1 in enumerate(all_atrib):
    for j, col2 in enumerate(all_atrib):
        if col1 == col2:
            corr_matrix.loc[col1, col2] = 1.0
            
        elif col1 in num_cols and col2 in num_cols:
            # Numérico vs numérico: Pearson
            corr_matrix.loc[col1, col2] = df_eda[col1].corr(df_eda[col2])
            
        elif col1 in cat_atrib and col2 in cat_atrib:
            # Categórico vs categórico: Cramér's V
            corr_matrix.loc[col1, col2] = cramers_v(df_eda[col1], df_eda[col2])
            
        else:
            # Categórico vs numérico: Correlation Ratio η
            cat_col = col1 if col1 in cat_atrib else col2
            num_col = col1 if col1 in num_atrib else col2
            corr_matrix.loc[col1, col2] = correlation_ratio(df_eda[cat_col], df_eda[num_col])

# ==========================================
# 4. VISUALIZACIÓN
# ==========================================
plt.figure(figsize=(12, 10))
# Usamos vmin=0 y vmax=1 porque Cramer y Eta van de 0 a 1 (aunque Pearson puede ir a -1, tomamos valor absoluto visualmente si prefieres)
sns.heatmap(corr_matrix.abs(), annot=True, cmap='Blues', fmt=".2f", linewidths=0.5)
plt.title("Matriz de Correlación Mixta (Pearson, V de Cramer, Eta $\eta$)")
plt.tight_layout()
plt.show()